# Create Monitoring Baseline
Establish baseline for fraud detection model monitoring

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
import pandas as pd
import time

In [ ]:
# Create monitor
monitor = DefaultModelMonitor(
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

In [ ]:
# Prepare baseline dataset (training data without target)
baseline_data = pd.read_csv(f's3://{bucket}/data/fraud_dataset.csv')
feature_columns = [col for col in baseline_data.columns 
                  if col not in ['is_fraud', 'transaction_id', 'timestamp']]
baseline_features = baseline_data[feature_columns]

# Save baseline data
baseline_s3_uri = f's3://{bucket}/baseline/baseline.csv'
baseline_features.to_csv(baseline_s3_uri, index=False)
print(f"Baseline data saved to: {baseline_s3_uri}")

In [ ]:
# Create baseline
baseline_job_name = f'fraud-baseline-{int(time.time())}'
baseline_job = monitor.suggest_baseline(
    baseline_dataset=baseline_s3_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f's3://{bucket}/baseline-results',
    job_name=baseline_job_name
)

print(f"Baseline job started: {baseline_job_name}")
baseline_job.wait()

In [ ]:
# Check baseline results
baseline_results = baseline_job.describe()
print(f"Baseline job status: {baseline_results['ProcessingJobStatus']}")
print(f"Output location: {baseline_results['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']}")